In [ ]:
#READ EXCEL
import pandas as pd
import numpy as np

df = pd.read_excel("Cleaned_FIFA_Data.xlsx")

print(df.head())


In [ ]:
# GET UNIQUE COUNTRIES
#nationalities = df["Nationality"]
#print(nationalities.head())

unique_nationalities = df["Nationality"].unique()

print(np.sort(unique_nationalities))


In [ ]:
# ENTER UNIQUE COUNTRIES
countries_df = pd.DataFrame(np.sort(unique_nationalities), columns= ["Nationality"])
countries_df.dropna()
countries_df.drop_duplicates()

print(countries_df) 

In [ ]:
# FIX COUNTRY NAMES
country_dictionary = {
    "St Lucia": "Saint Lucia",
    "St Kitts Nevis": "Saint Kitts and Nevis",
    "São Tomé & Príncipe": "São Tomé and Príncipe",
    "Antigua & Barbuda": "Antigua and Barbuda",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Brunei Darussalam": "Brunei",
    "Central African Rep.": "Central African Republic",
    "DR Congo": "Democratic Republic of the Congo",
    "Korea DPR": "North Korea",
    "Korea Republic": "South Korea",
    "China PR": "China",
    "FYR Macedonia": "North Macedonia"
}

countries_df["Fixed_Name"] = (countries_df["Nationality"].replace(country_dictionary))
countries_df["Search_Name"] = (countries_df["Fixed_Name"].str.replace(' ', '_'))

countries_df["Flag_URL"] = ""

print(countries_df)

In [ ]:
# TEST GET URL
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Portugal"

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

response = requests.get(url, headers=my_info_header)
#print(response.status_code)
#print(response.text)

soup = BeautifulSoup(response.text, "html.parser")

#print ("*****")
#print(soup.title)

website_images = []
for item in soup.find_all('img'):
    website_images.append(item["src"])
#print(website_images)

flag_images = []
for img in website_images:
    if "flag" in img.lower():
        flag_images.append(img)

#print(flag_images)

flag_url= flag_images[0]
#print(flag_url)

if flag_url.startswith("//"):
    flag_url = "https:" + flag_url

print(flag_url)


In [ ]:
#TEST DOWNLOAD IMAGE
my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

img_response = requests.get(flag_url, headers=my_info_header)

print(img_response.status_code)
print(img_response.headers["Content-Type"])

with open("country_images/portugal_flag.png", "wb") as f:
    f.write(img_response.content)


In [ ]:
# FUNCTIONS GET URL N DOWNLOAD IMAGE
import requests
from bs4 import BeautifulSoup

my_info_header = {
    "User-Agent": "MyWikipediaBot/1.0 (https://github.com/Vicky-Kounadi)"
}

def get_flag_url(country):
        # CONNECT
        url = f"https://en.wikipedia.org/wiki/{country}"

        response = requests.get(url, headers=my_info_header)

        soup = BeautifulSoup(response.text, "html.parser")

        # GET ALL IMAGES
        website_images = []
        for item in soup.find_all('img'):
            website_images.append(item["src"])

        # FIND FLAG IMAGES
        flag_images = []
        for img in website_images:
            if "flag" in img.lower():
                flag_images.append(img)

        # IF NO FLAG FOUND
        if len(flag_images) == 0:
            return None

        # USE THE FIRST ONE
        flag_url= flag_images[0]

        # FIX URL
        if flag_url.startswith("//"):
            flag_url = "https:" + flag_url

        #print(flag_url)

        return flag_url

def download_flag_image(url, country):
        # INVALID LINK
        if pd.isna(url) or not url:
            url= "https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Solid_black.svg/500px-Solid_black.svg.png"
       
        # DOWNLOAD IMAGES            
        img_response = requests.get(url, headers=my_info_header)

        print(img_response.status_code)
        print(img_response.headers["Content-Type"])

        with open(f"assets/country_images/{country}_flag.png", "wb") as f:
            f.write(img_response.content)



In [ ]:
# USE GET URL
countries_df["Flag_URL"] = countries_df["Search_Name"].apply(get_flag_url)

In [ ]:
# EXPORT DATA URL TO CSV
print(countries_df)

countries_df.to_csv("countries_flag_urls.csv", index=True)

In [ ]:
# USE DOWLOAD IMAGE
countries_df.apply(
    lambda rec: download_flag_image(rec["Flag_URL"], rec["Search_Name"]),
    axis=1
)

In [ ]:
# LOAD FIXED CSV N DOWNLOAD IMAGES
import pandas as pd

countries_fixed = pd.read_csv("countries_flag_urls.csv")

countries_fixed.apply(
    lambda rec: download_flag_image( rec["Flag_URL"], rec["Search_Name"]),
    axis=1
)